# 01 — S01 field structure and elastic background

This notebook performs the complete field-conditioning stage used by SAGE-AVO:

1. stream the real S01 SEG-Y into near/mid/far AVO and full structural stacks;
2. estimate PWD slope, structure-oriented seismic, and raw RGT;
3. repair RGT monotonicity explicitly for safe inversion while retaining the raw result;
4. read the real Petrel T6/T7 grids and five LAS wells;
5. convert depth horizons to time with spatially weighted well time-depth curves;
6. evaluate constant-RGT horizon picks as a diagnostic;
7. construct a local T6–T7 stratigraphic coordinate and interpolate reservoir DELTA and porosity with explicit support QC;
8. validate Vp/Vs/RHOB models by leave-one-well-out folds and fit the final field background;
9. write the canonical versioned artifact contract and manifests.

**Private inputs:** S01 SEG-Y, T6/T7 Petrel grids, and authorized LAS wells configured in ignored `configs/paths.yaml`.

**Outputs:** `data/s01data/{usable,attributes,derived,bundles}/v001`, plus a project-level `data/avo/s01/bundles/v001` manifest. Generated payloads remain ignored by Git.

**Runtime:** cached structural reruns are typically under two minutes on CPU; rebuilding the SEG-Y stacks takes several minutes. No GPU or Madagascar command-line execution is required. `segyio`, `pyseistr`, `lasio`, scikit-learn, and joblib are required (`pip install -e ".[field,notebooks]"`).

**Scientific scope:** Petrel depth→time surfaces preserve interpretation provenance, while explicitly labeled seismic-conformed T6/T7 surfaces define the property and elastic modeling interval. Support-confidence and extrapolation masks accompany every reservoir product. The exported DELTA, porosity, Vp, Vs, and density grids are field-conditioned structure-model backgrounds. `DELTA` is shaliness, so `P(sand) = 1 - DELTA`.


## 1. Configuration and canonical data contract

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "sage_avo").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the SAGE-AVO repository or notebooks directory.")


ROOT = find_project_root(Path.cwd())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Interactive notebooks can retain stale local package modules across edits.
# Purge SAGE-AVO modules so this cell always imports the current ROOT/src code.
for module_name in list(sys.modules):
    if module_name == "sage_avo" or module_name.startswith("sage_avo."):
        del sys.modules[module_name]

import json
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sage_avo.config import load_config, seed_everything
from sage_avo.data.field import (
    inspect_segy,
    load_field_line_stacks,
    save_field_line_stacks,
    stack_segy_line,
)
from sage_avo.data.interpretation import read_las_wells, read_petrel_points
from sage_avo.data.layout import DataLayout
from sage_avo.experiments.manifest import file_sha256, git_commit, write_json
from sage_avo.geology.field_conditioning import (
    blend_horizon_conditioned_background,
    build_horizon_conditioned_fields,
    build_horizon_conditioned_fields,
    build_reservoir_training_table,
    build_well_training_table,
    build_wheeler_fields,
    fit_grouped_elastic_models,
    interval_mask,
    predict_elastic_fields,
)
from sage_avo.structure.calibration import (
    build_well_horizon_table,
    calibrate_horizon_rgt,
    depth_surface_to_time,
    project_horizon_depth,
)
from sage_avo.structure.rgt import (
    estimate_pwd_rgt,
    load_pwd_rgt,
    monotonicity_report,
    repair_rgt_monotonicity,
    save_pwd_rgt,
)

required_api = [
    DataLayout,
    inspect_segy,
    load_field_line_stacks,
    read_las_wells,
    read_petrel_points,
    save_field_line_stacks,
    stack_segy_line,
    blend_horizon_conditioned_background,
    build_horizon_conditioned_fields,
    build_reservoir_training_table,
    build_well_training_table,
    build_wheeler_fields,
    fit_grouped_elastic_models,
    interval_mask,
    predict_elastic_fields,
    build_well_horizon_table,
    calibrate_horizon_rgt,
    depth_surface_to_time,
    project_horizon_depth,
    estimate_pwd_rgt,
    load_pwd_rgt,
    monotonicity_report,
    repair_rgt_monotonicity,
    save_pwd_rgt,
]
missing_api = [getattr(item, "__name__", repr(item)) for item in required_api if item is None]
if missing_api:
    raise ImportError(f"Local sage_avo API is incomplete: {missing_api}")

workflow_path = ROOT / "configs" / "structure_model_s01.yaml"
paths_path = ROOT / "configs" / "paths.yaml"
if not paths_path.is_file():
    raise FileNotFoundError("Copy configs/paths.example.yaml to configs/paths.yaml and enter authorized paths.")

workflow = load_config(workflow_path)
paths = load_config(paths_path)
dataset_cfg = workflow["dataset"]
layout = DataLayout(
    work_root=Path(paths["work_data_root"]),
    dataset=dataset_cfg["id"],
    version=dataset_cfg["version"],
    raw_root=Path(paths["s01_raw_root"]),
)
layout.ensure_outputs("usable", "attributes", "derived", "bundles")
seed_everything(int(workflow["elastic_model"]["seed"]))

field_segy = Path(paths["field_segy"]).expanduser()
horizon_paths = {"T6": Path(paths["horizon_t6"]).expanduser(), "T7": Path(paths["horizon_t7"]).expanduser()}
well_paths = [Path(value).expanduser() for value in paths["well_las_files"]]

display(pd.Series(layout.to_dict(), name="canonical S01 layout"))


In [ ]:
required_inputs = {"SEG-Y": field_segy, **{f"{name} horizon": path for name, path in horizon_paths.items()}}
required_inputs.update({f"well {index + 1}": path for index, path in enumerate(well_paths)})
input_status = pd.DataFrame(
    [{"input": name, "path": str(path), "available": path.is_file(), "size_MiB": round(path.stat().st_size / 2**20, 2) if path.is_file() else np.nan}
     for name, path in required_inputs.items()]
)
display(input_status)
if not input_status["available"].all():
    missing = input_status.loc[~input_status["available"], "path"].tolist()
    raise FileNotFoundError(f"Missing required S01 inputs: {missing}")

## 2. Inspect the real SEG-Y and build field stacks

In [ ]:
segy_cfg = workflow["segy"]
summary = inspect_segy(field_segy, angle_header=segy_cfg["angle_header"])
configured_angles = sorted({angle for limits in segy_cfg["bands_degrees"].values() for angle in range(int(limits[0]), int(limits[1]) + 1)})
if list(summary.angle_header_values) != configured_angles:
    raise ValueError("Observed offset-word values do not match the configured 3–45 degree bins.")
display(pd.Series(summary.to_dict(), name="S01 SEG-Y"))
print("Numerical angle-bin check passed. Header semantics still depend on the documented local export convention.")

In [ ]:
cache_dir = layout.derived / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
stack_cache = cache_dir / "field_line_stacks.npz"
REBUILD_SEGY_STACKS = False

if stack_cache.is_file() and not REBUILD_SEGY_STACKS:
    line = load_field_line_stacks(stack_cache)
    stack_source = "canonical cache"
else:
    line = stack_segy_line(
        field_segy,
        bands_degrees=segy_cfg["bands_degrees"],
        time_window_ms=tuple(segy_cfg["time_window_ms"]),
        midpoint_y_max=segy_cfg["midpoint_y_max"],
        angle_header=segy_cfg["angle_header"],
        angle_header_semantics=segy_cfg["angle_header_semantics"],
        robust_clip_percentiles=tuple(segy_cfg["robust_clip_percentiles"]),
        chunk_traces=int(segy_cfg["chunk_traces"]),
    )
    save_field_line_stacks(stack_cache, line)
    stack_source = "SEG-Y"

print(f"Source: {stack_source}; AVO shape [band,time,CDP] = {line.avo.shape}")
display(pd.DataFrame({
    "limits_deg": line.band_limits_degrees,
    "min_fold": line.band_fold.min(axis=1),
    "median_fold": np.median(line.band_fold, axis=1).astype(int),
    "max_fold": line.band_fold.max(axis=1),
}, index=line.band_names))

In [ ]:
extent = [line.cdps[0], line.cdps[-1], line.time_ms[-1], line.time_ms[0]]
avo_limit = float(np.percentile(np.abs(line.avo[np.isfinite(line.avo)]), 99.0))
fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True, constrained_layout=True)
for axis, image, title in zip(axes, [*line.avo, line.seismic_structure], [*(f"{n.capitalize()} AVO" for n in line.band_names), "Structural stack"]):
    limit = avo_limit if "AVO" in title else 3.0
    panel = axis.imshow(image, cmap="gray", aspect="auto", extent=extent, vmin=-limit, vmax=limit)
    axis.set(title=title, xlabel="CDP")
    fig.colorbar(panel, ax=axis, fraction=0.046, pad=0.03)
axes[0].set_ylabel("Two-way time (ms)")
plt.show()

**Figure explanation — AVO and structural stacks.** Near, mid, and far panels are angle-band averages from the field SEG-Y and show qualitative amplitude-versus-angle behavior. The structural stack uses all accepted traces and is independently normalized to emphasize reflector continuity. Horizontal position is CDP and time increases downward.


## 3. PWD/RGT and explicit monotonic repair

In [ ]:
structure_cfg = workflow["structure"]
rgt_cache = cache_dir / "pwd_rgt.npz"
REBUILD_RGT = False
if rgt_cache.is_file() and not REBUILD_RGT:
    structure = load_pwd_rgt(rgt_cache)
    rgt_source = "canonical cache"
else:
    structure = estimate_pwd_rgt(
        line.seismic_structure, line.time_ms,
        gaussian_sigma=tuple(structure_cfg["gaussian_sigma"]),
        dip_order=int(structure_cfg["dip_order"]),
        dip_iterations=int(structure_cfg["dip_iterations"]),
        dip_rect=tuple(structure_cfg["dip_rect"]),
        structure_radius=int(structure_cfg["structure_radius"]),
        structure_order=int(structure_cfg["structure_order"]),
        structure_epsilon=float(structure_cfg["structure_epsilon"]),
        rgt_epsilon=float(structure_cfg["rgt_epsilon"]),
    )
    save_pwd_rgt(rgt_cache, structure)
    rgt_source = "two-pass PySeistr"

raw_rgt_qc = monotonicity_report(structure.rgt, tolerance=float(structure_cfg["monotonicity_tolerance"]))
rgt_tau, repair_qc = repair_rgt_monotonicity(
    structure.rgt, minimum_step=float(structure_cfg["monotonic_repair_minimum_step"])
)
repaired_rgt_qc = monotonicity_report(rgt_tau, tolerance=0.0)
display(pd.DataFrame([{"product": "raw", **raw_rgt_qc}, {"product": "isotonic repair", **repaired_rgt_qc, **repair_qc}]))
print(f"RGT source: {rgt_source}. Raw RGT is retained; repaired RGT is the downstream coordinate.")

In [ ]:
dip_limit = float(np.percentile(np.abs(structure.dip), 99.0))
fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True, constrained_layout=True)
panels = [
    (structure.structure_oriented_seismic, "Structure-oriented stack", "gray", None, None),
    (structure.dip, "Refined local slope", "RdBu_r", -dip_limit, dip_limit),
    (structure.rgt, "Raw RGT", "viridis", None, None),
    (rgt_tau, "Monotonic RGT", "viridis", None, None),
]
for axis, (image, title, cmap, vmin, vmax) in zip(axes, panels):
    panel = axis.imshow(image, cmap=cmap, aspect="auto", extent=extent, vmin=vmin, vmax=vmax)
    axis.set(title=title, xlabel="CDP")
    fig.colorbar(panel, ax=axis, fraction=0.046, pad=0.03)
axes[0].set_ylabel("Two-way time (ms)")
plt.show()

**Figure explanation — structure and RGT.** The structure-oriented stack suppresses cross-reflector noise before slope estimation. Red/blue local slope indicates opposite reflector dip directions. Raw RGT is the direct PWD integration result; monotonic RGT is its trace-by-trace isotonic repair. Colors are relative geologic coordinates, not ages. Differences between the last two panels show where repair was required.


## 4. Real horizons and wells

In [ ]:
horizons = {name: read_petrel_points(path) for name, path in horizon_paths.items()}
wells = read_las_wells(well_paths)
well_qc = pd.DataFrame([{"WELL": well.name, "X": well.x, "Y": well.y, **well.qc} for well in wells])
display(pd.DataFrame([{"horizon": name, "points": len(frame), "z_min_m": frame["Z"].min(), "z_max_m": frame["Z"].max()} for name, frame in horizons.items()]))
display(well_qc)
print("Convention check: DELTA is shaliness; P(sand)=1-DELTA. PORO values are already fractions despite the LAS '%' label.")

In [ ]:
horizon_cfg = workflow["horizons"]
t6_depth, t6_projection_distance = project_horizon_depth(horizons["T6"], line.line_xy, max_distance=float(horizon_cfg["projection_max_distance_m"]))
t7_depth, t7_projection_distance = project_horizon_depth(horizons["T7"], line.line_xy, max_distance=float(horizon_cfg["projection_max_distance_m"]))
t6_time = depth_surface_to_time(t6_depth, line.line_xy, wells, distance_scale=float(horizon_cfg["depth_time_distance_scale_m"]))
t7_time = depth_surface_to_time(t7_depth, line.line_xy, wells, distance_scale=float(horizon_cfg["depth_time_distance_scale_m"]))
well_table = build_well_horizon_table(wells, horizons["T6"], horizons["T7"], line.line_xy, line.cdps)
display(well_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].plot(line.cdps, t6_depth, label="T6"); axes[0].plot(line.cdps, t7_depth, label="T7")
axes[0].invert_yaxis(); axes[0].set(title="Projected depth horizons", xlabel="CDP", ylabel="Depth (m)"); axes[0].legend()
axes[1].imshow(line.seismic_structure, cmap="gray", aspect="auto", extent=extent, vmin=-3, vmax=3)
axes[1].plot(line.cdps, t6_time, label="T6 depth→time"); axes[1].plot(line.cdps, t7_time, label="T7 depth→time")
axes[1].set(title="Primary interpreted time surfaces", xlabel="CDP", ylabel="TWT (ms)"); axes[1].legend()
plt.show()

**Figure explanation — interpreted horizons.** The left panel is the Petrel T6/T7 depth projection onto the 2-D line. The right panel converts those depths to TWT using spatially weighted well depth–time curves and overlays them on seismic. These are the primary interpreted surfaces. Their convergence at the far-right edge exists in the source grids; columns thinner than the configured minimum are excluded from reservoir modeling.


## 5. Seismic-conformed modeling horizons and alignment QC

In [ ]:
t6_calibration = calibrate_horizon_rgt("T6", rgt_tau, line.time_ms, t6_time, well_table)
t7_calibration = calibrate_horizon_rgt("T7", rgt_tau, line.time_ms, t7_time, well_table)
horizon_qc = pd.DataFrame([{"horizon": result.name, "rgt_reference": result.rgt_reference, **result.qc} for result in (t6_calibration, t7_calibration)])
diagnostic_limit = float(horizon_cfg["rgt_diagnostic_max_rmse_ms"])
horizon_qc["meets_alignment_threshold"] = horizon_qc["input_rmse_ms"] <= diagnostic_limit
display(horizon_qc)

modeling_surface_policy = str(horizon_cfg["modeling_surface_policy"])
if modeling_surface_policy != "seismic_conformed_constant_rgt":
    raise ValueError(f"Unsupported modeling surface policy: {modeling_surface_policy}")
model_t6_time = t6_calibration.picked_time_ms.copy()
model_t7_time = t7_calibration.picked_time_ms.copy()
if not np.all(model_t7_time > model_t6_time):
    raise ValueError("Seismic-conformed T7 must remain below T6 on every trace")
model_horizon_qc = {
    "policy": modeling_surface_policy,
    "t6_interpreted_vs_model_rmse_ms": float(t6_calibration.qc["input_rmse_ms"]),
    "t7_interpreted_vs_model_rmse_ms": float(t7_calibration.qc["input_rmse_ms"]),
    "minimum_model_interval_ms": float(np.min(model_t7_time - model_t6_time)),
}
display(pd.Series(model_horizon_qc, name="modeling-surface policy"))

fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(line.seismic_structure, cmap="gray", aspect="auto", extent=extent, vmin=-3, vmax=3)
ax.plot(line.cdps, t6_time, "C1--", label="T6 depth-derived interpretation"); ax.plot(line.cdps, t6_calibration.picked_time_ms, "C1", label="T6 seismic-conformed model")
ax.plot(line.cdps, t7_time, "C0--", label="T7 depth-derived interpretation"); ax.plot(line.cdps, t7_calibration.picked_time_ms, "C0", label="T7 seismic-conformed model")
ax.set(xlabel="CDP", ylabel="TWT (ms)", title="Horizon roles and seismic alignment QC")
ax.legend(ncol=2)
plt.show()


**Figure explanation — horizon roles and alignment.** Dashed curves are the Petrel depth→time interpretation surfaces; solid curves are the seismic-conformed constant-RGT modeling surfaces. The table reports their RMSE separation and the configured alignment threshold. The depth-derived surfaces preserve interpretation provenance, while the seismic-conformed surfaces define the modeling boundaries used by the facies and elastic backgrounds.


## 6. Horizon-conditioned reservoir properties

The depth-derived surfaces preserve interpretation provenance, while the seismic-conformed T6/T7 curves from Section 5 define the modeling reservoir. A local coordinate is constructed with a stable normalized-time backbone and a smoothed RGT steering term:

\[
s=(1-\alpha)u+\alpha\widetilde{\tau},\qquad
u=\frac{t-T6_model(x)}{T7_model(x)-T6_model(x)},\qquad \alpha=0.35.
\]

Both components are normalized to zero on T6 and one on T7. The time component prevents repaired-RGT steps from creating blocks; the RGT component retains moderate internal structural steering. Only wells with valid T6 and T7 ties train the reservoir model. Absolute distance weights produce a continuous support-confidence field rather than implying that normalized weights are equally trustworthy everywhere.


In [ ]:
training = build_well_training_table(wells, well_table, line.time_ms, rgt_tau)
training_clean = training.dropna(subset=["DELTA", "PORO", "VP", "VS", "RHOB", "RGT"])
display(training_clean.groupby("WELL").size().rename("seismic-time samples").to_frame())

# Regional fields are finite numerical backgrounds made from all five wells.
wheeler_cfg = workflow["wheeler"]
regional_wheeler = build_wheeler_fields(
    training, wells, line.line_xy, rgt_tau,
    n_rgt=int(wheeler_cfg["n_rgt"]),
    lateral_distance_scale_m=float(wheeler_cfg["lateral_distance_scale_m"]),
    smooth_sigma=tuple(wheeler_cfg["smooth_sigma"]),
)

# Reservoir fields use only wells with complete T6/T7 ties and the stabilized
# local stratigraphic coordinate described above.
condition_cfg = workflow["horizon_conditioning"]
reservoir_fields = build_horizon_conditioned_fields(
    training, wells, well_table, line.line_xy, line.time_ms, rgt_tau, model_t6_time, model_t7_time,
    n_strat=int(condition_cfg["n_strat"]),
    lateral_distance_scale_m=float(condition_cfg["lateral_distance_scale_m"]),
    max_support_distance_m=float(condition_cfg["max_support_distance_m"]),
    minimum_support_confidence=float(condition_cfg["minimum_support_confidence"]),
    minimum_interval_ms=float(condition_cfg["minimum_interval_ms"]),
    minimum_rgt_separation=float(condition_cfg["minimum_rgt_separation"]),
    rgt_steering_weight=float(condition_cfg["rgt_steering_weight"]),
    coordinate_smooth_sigma=tuple(condition_cfg["coordinate_smooth_sigma"]),
    smooth_sigma=tuple(condition_cfg["smooth_sigma"]),
)

reservoir_mask = reservoir_fields.reservoir_mask
reservoir_mask_interpreted = interval_mask(line.time_ms, t6_time, t7_time)
property_support_mask = reservoir_fields.support_mask
property_support_confidence = reservoir_fields.support_confidence
property_extrapolated_mask = (
    reservoir_mask.astype(bool) & ~property_support_mask.astype(bool)
).astype(np.uint8)

# Smoothstep tapering makes the reservoir contribution zero exactly on T6/T7,
# avoiding hard seams with the regional field. Weakly supported locations are
# downweighted continuously rather than cut off abruptly.
delta_model, property_blend_weight = blend_horizon_conditioned_background(
    regional_wheeler.delta_time,
    reservoir_fields.delta_time,
    reservoir_fields.strat_coordinate,
    property_support_confidence,
    edge_fraction=float(condition_cfg["composite_edge_fraction"]),
)
porosity_model, porosity_blend_weight = blend_horizon_conditioned_background(
    regional_wheeler.porosity_time,
    reservoir_fields.porosity_time,
    reservoir_fields.strat_coordinate,
    property_support_confidence,
    edge_fraction=float(condition_cfg["composite_edge_fraction"]),
)
assert np.allclose(property_blend_weight, porosity_blend_weight)
sand_probability_model = 1.0 - delta_model

valid_columns = reservoir_fields.geometry_valid.astype(bool)
# The normalized-domain construction explicitly sets its continuous boundary
# curves to s(T6)=0 and s(T7)=1. Raster samples need not land exactly on either
# interpreted time, so the invariant is tested in the core unit test rather
# than by interpolating through NaNs/pixel centers here.
assert np.allclose(delta_model + sand_probability_model, 1.0)

reservoir_pixels = int(reservoir_mask.sum())
supported_pixels = int(property_support_mask.sum())
property_qc = {
    "wells_used": list(reservoir_fields.wells_used),
    "well_distance_to_line_m": {
        str(row.WELL): float(row.DISTANCE_TO_LINE_M)
        for row in well_table[well_table["WELL"].isin(reservoir_fields.wells_used)].itertuples()
    },
    "modeling_surface_policy": modeling_surface_policy,
    "geometry_valid_columns": int(valid_columns.sum()),
    "geometry_total_columns": int(valid_columns.size),
    "reservoir_pixels": reservoir_pixels,
    "supported_pixels": supported_pixels,
    "supported_fraction": float(supported_pixels / max(reservoir_pixels, 1)),
    "support_confidence_min": float(property_support_confidence[reservoir_mask.astype(bool)].min()),
    "support_confidence_median": float(np.median(property_support_confidence[reservoir_mask.astype(bool)])),
    "support_confidence_max": float(property_support_confidence.max()),
    "nearest_well_distance_m_min": float(reservoir_fields.nearest_well_distance_m.min()),
    "nearest_well_distance_m_max": float(reservoir_fields.nearest_well_distance_m.max()),
    "t6_coordinate_boundary": 0.0,
    "t7_coordinate_boundary": 1.0,
}
display(pd.Series(property_qc, name="horizon-conditioned property QC"))
print(
    f"Reservoir DELTA {np.nanmin(reservoir_fields.delta_time):.3f}–{np.nanmax(reservoir_fields.delta_time):.3f}; "
    f"P(sand) {np.nanmin(reservoir_fields.sand_probability_time):.3f}–{np.nanmax(reservoir_fields.sand_probability_time):.3f}; "
    f"porosity {np.nanmin(reservoir_fields.porosity_time):.3f}–{np.nanmax(reservoir_fields.porosity_time):.3f}"
)


In [ ]:
reservoir_valid = reservoir_mask.astype(bool)
confidence = property_support_confidence
display_alpha = np.where(reservoir_valid, 0.12 + 0.88 * confidence, 0.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True, constrained_layout=True)
panels = [
    (reservoir_fields.delta_time, "DELTA: horizon-conditioned reservoir", "viridis", (0, 1), display_alpha),
    (reservoir_fields.sand_probability_time, "P(sand) = 1 − DELTA", "viridis", (0, 1), display_alpha),
    (reservoir_fields.porosity_time, "Porosity: horizon-conditioned reservoir", "viridis", (0, 0.08), display_alpha),
    (confidence, "Absolute well-support confidence", "magma", (0, 1), np.where(reservoir_valid, 1.0, 0.0)),
]
for panel_index, (axis, (image, title, cmap_name, limits, alpha)) in enumerate(zip(axes, panels)):
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("0.85")
    artist = axis.imshow(
        np.ma.masked_invalid(image), aspect="auto", extent=extent,
        cmap=cmap, vmin=limits[0], vmax=limits[1], alpha=alpha,
    )
    axis.plot(line.cdps, model_t6_time, "k-", lw=1.4, label="T6 model (seismic-conformed)")
    axis.plot(line.cdps, model_t7_time, "k--", lw=1.4, label="T7 model (seismic-conformed)")
    axis.plot(line.cdps, t6_time, color="tab:red", ls=":", lw=0.8, label="T6 depth→time")
    axis.plot(line.cdps, t7_time, color="tab:cyan", ls=":", lw=0.8, label="T7 depth→time")
    tied_rows = well_table[well_table["WELL"].isin(reservoir_fields.wells_used)]
    for _, row in tied_rows.iterrows():
        midpoint = np.nanmean([row["T6_TWT_MS"], row["T7_TWT_MS"]])
        axis.plot(row["MATCHED_CDP"], midpoint, "wo", mec="k", ms=5)
        if panel_index == 3:
            axis.annotate(row["WELL"], (row["MATCHED_CDP"], midpoint), xytext=(4, 4),
                          textcoords="offset points", fontsize=8, color="white")
    axis.set(title=title, xlabel="CDP")
    fig.colorbar(artist, ax=axis, fraction=0.046, pad=0.03)
axes[0].set_ylabel("TWT (ms)")
axes[0].legend(loc="lower left", fontsize=8)
plt.show()


**Figure explanation — reservoir properties and the three circles.** Black solid/dashed curves are the seismic-conformed T6/T7 modeling boundaries; thin red/cyan dotted curves are the depth-derived interpretation surfaces. The circles mark the nearest-line locations of T732, T761, and T762—the three wells with complete usable T6/T7 ties—and are placed at each well's interval midpoint as location markers. DELTA is shaliness and P(sand) is its exact complement. Opacity follows absolute well-support confidence, so pale left-side values identify extrapolation. Property layering follows the seismic-conformed modeling interval.


## 7. Regional and reservoir elastic models

Vp, Vs, and density require two distinct models because global RGT and the interpreted reservoir coordinate are not equivalent:

- LAS compressional and shear slowness are converted to `VP = 304800 / DT` and `VS = 304800 / SDT`; RHOB is the measured density log.
- The **regional model** is trained on all five wells with `[DELTA, PORO, RGT]`. It supplies the complete field outside T6–T7.
- The **reservoir model** is trained only on T732, T761, and T762—the wells with complete T6/T7 ties—with `[DELTA, PORO, STRAT_FRACTION]`. It therefore cannot pull reservoir predictions back onto incompatible global-RGT layers.
- Leave-one-well-out validation is performed separately for both scopes.
- At T6 and T7 the reservoir weight is exactly zero. Inside the interval, the reservoir elastic prediction enters with the same smooth boundary taper and absolute support confidence used for the property model.
- The final grids are supervised, well-log-conditioned elastic backgrounds for downstream AVO modeling and inversion.


In [ ]:
elastic_cfg = workflow["elastic_model"]

regional_model_set = fit_grouped_elastic_models(
    training,
    feature_columns=("DELTA", "PORO", "RGT"),
    seed=int(elastic_cfg["seed"]),
    n_estimators=int(elastic_cfg["n_estimators"]),
    min_samples_leaf=int(elastic_cfg["min_samples_leaf"]),
)

reservoir_training = build_reservoir_training_table(
    training,
    well_table,
    rgt_steering_weight=float(condition_cfg["rgt_steering_weight"]),
    minimum_interval_ms=float(condition_cfg["minimum_interval_ms"]),
    minimum_rgt_separation=float(condition_cfg["minimum_rgt_separation"]),
)
reservoir_model_set = fit_grouped_elastic_models(
    reservoir_training,
    feature_columns=("DELTA", "PORO", "STRAT_FRACTION"),
    seed=int(elastic_cfg["seed"]),
    n_estimators=int(elastic_cfg["n_estimators"]),
    min_samples_leaf=int(elastic_cfg["min_samples_leaf"]),
)

elastic_cv = pd.concat(
    [
        regional_model_set.cross_validation.assign(model_scope="regional_all_wells"),
        reservoir_model_set.cross_validation.assign(model_scope="reservoir_tied_wells"),
    ],
    ignore_index=True,
)
display(elastic_cv)
display(
    elastic_cv.groupby(["model_scope", "target"])[["rmse", "mae", "r2"]]
    .agg(["mean", "std"])
)

regional_elastic_background = predict_elastic_fields(
    regional_model_set,
    regional_wheeler.delta_time,
    regional_wheeler.porosity_time,
    rgt_tau,
)
reservoir_elastic_background = predict_elastic_fields(
    reservoir_model_set,
    reservoir_fields.delta_time,
    reservoir_fields.porosity_time,
    reservoir_fields.strat_coordinate,
)

elastic_components = []
elastic_weights = []
for regional_property, reservoir_property in zip(
    regional_elastic_background, reservoir_elastic_background
):
    composite, weight = blend_horizon_conditioned_background(
        regional_property,
        reservoir_property,
        reservoir_fields.strat_coordinate,
        property_support_confidence,
        edge_fraction=float(condition_cfg["composite_edge_fraction"]),
    )
    elastic_components.append(composite)
    elastic_weights.append(weight)
elastic_background = np.asarray(elastic_components, dtype=np.float32)
elastic_blend_weight = np.asarray(elastic_weights, dtype=np.float32)
assert np.allclose(elastic_blend_weight, property_blend_weight[None, :, :])

vp, vs, rhob = elastic_background
print(f"Regional elastic background = {regional_elastic_background.shape}")
print(f"Reservoir elastic background = {reservoir_elastic_background.shape}")
print(f"Final blended elastic background = {elastic_background.shape}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True, constrained_layout=True)
for axis, image, title, unit in zip(axes, elastic_background, ["Vp", "Vs", "Density"], ["m/s", "m/s", "g/cc"]):
    low, high = np.percentile(image, [1, 99])
    panel = axis.imshow(image, aspect="auto", extent=extent, cmap="viridis", vmin=low, vmax=high)
    axis.plot(line.cdps, model_t6_time, "w-", lw=1.4, label="T6 model")
    axis.plot(line.cdps, model_t7_time, "w--", lw=1.4, label="T7 model")
    axis.plot(line.cdps, t6_time, color="tab:red", ls=":", lw=0.7, label="T6 depth→time")
    axis.plot(line.cdps, t7_time, color="tab:cyan", ls=":", lw=0.7, label="T7 depth→time")
    axis.set(title=f"{title} horizon-conditioned composite background", xlabel="CDP")
    colorbar = fig.colorbar(panel, ax=axis, fraction=0.046, pad=0.03); colorbar.set_label(unit)
axes[0].set_ylabel("TWT (ms)")
axes[0].legend(loc="upper right", fontsize=7)
plt.show()


**Figure explanation — Vp, Vs, and density.** White solid/dashed curves are the seismic-conformed modeling T6/T7 and align with the elastic layering; red/cyan dotted curves preserve the independently depth-converted Petrel interpretations. Outside the modeling interval, the panels show the five-well regional Random Forest using global RGT. Inside, the three-tied-well reservoir forest uses local stratigraphic fraction, joined with a confidence-weighted taper. The panels are the exported field-conditioned elastic backgrounds.


## 8. Export the canonical artifact contract

In [ ]:
real_avo_dir = layout.usable / "real_avo"
well_dir = layout.usable / "wells"
horizon_dir = layout.attributes / "horizons"
qc_dir = layout.derived / "qc"
rf_dir = layout.dataset_root / "derived" / f"rf_models_{layout.version}"
for directory in (real_avo_dir, well_dir, horizon_dir, qc_dir, rf_dir):
    directory.mkdir(parents=True, exist_ok=True)

usable_arrays = {
    "seis_struct.npy": line.seismic_structure,
    "seis_struct_oriented.npy": structure.structure_oriented_seismic,
    "reg_t.npy": line.time_ms,
    "good_cdps.npy": line.cdps,
    "line_xy.npy": line.line_xy,
    "delta.npy": delta_model,
    "sandprob.npy": sand_probability_model,
    "poro.npy": porosity_model,
    "delta_reservoir.npy": reservoir_fields.delta_time,
    "sandprob_reservoir.npy": reservoir_fields.sand_probability_time,
    "poro_reservoir.npy": reservoir_fields.porosity_time,
    "property_support_mask.npy": property_support_mask,
    "property_support_confidence.npy": property_support_confidence,
    "property_blend_weight.npy": property_blend_weight,
    "property_extrapolated_mask.npy": property_extrapolated_mask,
    "reservoir_geometry_valid.npy": reservoir_fields.geometry_valid,
    "nearest_property_well_distance_m.npy": reservoir_fields.nearest_well_distance_m,
    "effective_property_well_count.npy": reservoir_fields.effective_well_count,
    "vp.npy": vp,
    "vs.npy": vs,
    "rhob.npy": rhob,
    "elastic_background.npy": elastic_background,
    "elastic_background_regional.npy": regional_elastic_background,
    "elastic_background_reservoir.npy": reservoir_elastic_background,
    "elastic_blend_weight.npy": elastic_blend_weight,
    "reservoir_mask.npy": reservoir_mask,
    "reservoir_mask_interpreted.npy": reservoir_mask_interpreted,
}
for name, array in usable_arrays.items(): np.save(layout.usable / name, np.asarray(array))
avo_arrays = {"AVO_stack3_real_raw.npy": np.moveaxis(line.avo, 0, -1), "AVO_low_real.npy": line.avo[0], "AVO_mid_real.npy": line.avo[1], "AVO_high_real.npy": line.avo[2]}
for name, array in avo_arrays.items(): np.save(real_avo_dir / name, np.asarray(array))

attribute_arrays = {
    "dip.npy": structure.dip, "rgt_raw.npy": structure.rgt, "rgt_tau.npy": rgt_tau,
    "fold.npy": line.structure_fold, "band_fold.npy": line.band_fold,
    "strat_fraction_t6_t7.npy": reservoir_fields.strat_coordinate,
    "strat_fraction_t6_t7.npy": reservoir_fields.strat_coordinate,
}
for name, array in attribute_arrays.items(): np.save(layout.attributes / name, np.asarray(array))
horizon_arrays = {
    "t6_depth_m.npy": t6_depth, "t7_depth_m.npy": t7_depth,
    "t6_time_interpreted_ms.npy": t6_time, "t7_time_interpreted_ms.npy": t7_time,
    "t6_rgt_diagnostic_ms.npy": t6_calibration.picked_time_ms,
    "t7_rgt_diagnostic_ms.npy": t7_calibration.picked_time_ms,
    "t6_model_seismic_conformed_ms.npy": model_t6_time,
    "t7_model_seismic_conformed_ms.npy": model_t7_time,
}
for name, array in horizon_arrays.items(): np.save(horizon_dir / name, np.asarray(array))

well_table.to_csv(layout.usable / "df_well.csv", index=False)
for well in wells: well.logs.to_csv(well_dir / f"{well.name}.csv", index=False)
elastic_cv.to_csv(qc_dir / "elastic_leave_one_well_out.csv", index=False)
horizon_qc.to_csv(qc_dir / "horizon_rgt_diagnostic.csv", index=False)
well_qc.to_csv(qc_dir / "well_time_qc.csv", index=False)
pd.DataFrame([property_qc]).to_json(qc_dir / "horizon_conditioned_property_qc.json", orient="records", indent=2)
pd.DataFrame([property_qc]).to_json(qc_dir / "horizon_conditioned_property_qc.json", orient="records", indent=2)
for target, model in regional_model_set.models.items():
    joblib.dump(model, rf_dir / f"rf_{target.lower()}.joblib")
    joblib.dump(model, rf_dir / f"rf_regional_{target.lower()}.joblib")
for target, model in reservoir_model_set.models.items():
    joblib.dump(model, rf_dir / f"rf_reservoir_{target.lower()}.joblib")
joblib.dump(regional_model_set, rf_dir / "elastic_model_set.joblib")
joblib.dump(regional_model_set, rf_dir / "elastic_model_set_regional.joblib")
joblib.dump(reservoir_model_set, rf_dir / "elastic_model_set_reservoir.joblib")
print("Canonical arrays, tables, and models written.")


In [ ]:
all_arrays = {
    **{str((layout.usable / name).relative_to(layout.dataset_root)): array for name, array in usable_arrays.items()},
    **{str((real_avo_dir / name).relative_to(layout.dataset_root)): array for name, array in avo_arrays.items()},
    **{str((layout.attributes / name).relative_to(layout.dataset_root)): array for name, array in attribute_arrays.items()},
    **{str((horizon_dir / name).relative_to(layout.dataset_root)): array for name, array in horizon_arrays.items()},
}
horizon_alignment_pass = bool(horizon_qc["meets_alignment_threshold"].all())
manifest = {
    "schema_version": 6,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": dataset_cfg,
    "layout": layout.to_dict(),
    "status": {
        "segy_stacking": "complete",
        "pwd_rgt_raw": "complete_with_monotonicity_warning" if raw_rgt_qc["fraction_bad"] > structure_cfg["max_fraction_bad"] else "complete",
        "rgt_isotonic_repair": "complete",
        "horizon_depth_time_conversion": "complete",
        "horizon_alignment_qc": "within_threshold" if horizon_alignment_pass else "review_interpretation_model_separation",
        "modeling_horizon_policy": modeling_surface_policy,
        "regional_rgt_property_background": "complete",
        "horizon_conditioned_reservoir_properties": "complete_with_explicit_support_mask",
        "regional_elastic_background": "complete_leave_one_well_out_validated",
        "reservoir_elastic_background": "complete_tied_well_leave_one_well_out_validated",
        "elastic_background": "complete_confidence_weighted_composite",
    },
    "source_inputs": {
        "segy": {**summary.to_dict(), "size_bytes": field_segy.stat().st_size, "mtime_ns": field_segy.stat().st_mtime_ns, "angle_header_semantics": segy_cfg["angle_header_semantics"]},
        "horizons": {name: {"path": str(path), "sha256": file_sha256(path)} for name, path in horizon_paths.items()},
        "wells": {path.stem: {"path": str(path), "sha256": file_sha256(path)} for path in well_paths},
    },
    "configuration": {"file": workflow_path.name, "sha256": file_sha256(workflow_path), "parameters": workflow},
    "code": {"git_commit": git_commit(ROOT)},
    "conventions": {"DELTA": "shaliness", "sand_probability": "1 - DELTA", "PORO": "fraction", "elastic_order": ["VP", "VS", "RHOB"]},
    "qc": {"raw_rgt": raw_rgt_qc, "rgt_repair": repair_qc, "horizons": horizon_qc.to_dict(orient="records"), "modeling_horizons": model_horizon_qc, "horizon_conditioned_properties": property_qc, "elastic_cv": elastic_cv.to_dict(orient="records")},
    "arrays": {name: {"shape": list(np.shape(array)), "dtype": str(np.asarray(array).dtype)} for name, array in all_arrays.items()},
    "limitations": [
        "The offset-word angle semantics depend on the local export convention.",
        "The raw RGT required a documented isotonic monotonicity repair.",
        "Depth-derived Petrel horizons preserve interpretation provenance; seismic-conformed constant-RGT surfaces are explicitly labeled modeling boundaries.",
        "Only wells with finite T6 and T7 time ties condition the reservoir property model.",
        "Support-confidence and blend-weight arrays identify weakly supported reservoir values and their contribution to composite backgrounds.",
        "Two of the three completely tied property wells are more than one kilometre from the 2-D line, so 3-D projection uncertainty remains material.",
        "Columns where T6–T7 thickness is below the configured minimum are excluded rather than forced open.",
        "LAS DEPTH and Petrel Z are assumed to share a compatible vertical datum; deviation-survey/TVDSS confirmation remains required.",
        "The reservoir elastic model has only three completely tied wells; its grouped validation is reported separately from the five-well regional model.",
        "Elastic grids are confidence-weighted regional/reservoir modeling backgrounds for downstream AVO analysis.",
        "Five wells condition the field model; leave-one-well-out scores quantify within-field well generalization.",
    ],
}
write_json(layout.bundles / "manifest.json", manifest)
project_bundle = ROOT / "data" / "avo" / "s01" / "bundles" / layout.version
project_bundle.mkdir(parents=True, exist_ok=True)
write_json(project_bundle / "manifest.json", {
    "schema_version": 1,
    "dataset_manifest": str((layout.bundles / "manifest.json").relative_to(ROOT / "data")),
    "dataset": dataset_cfg,
    "status": manifest["status"],
})
print(json.dumps(manifest["status"], indent=2))


In [ ]:
artifact_summary = pd.DataFrame([
    {"artifact": name, "shape": str(tuple(details["shape"])), "dtype": details["dtype"]}
    for name, details in manifest["arrays"].items()
])
display(artifact_summary)
print(f"Dataset manifest: {layout.bundles / 'manifest.json'}")
print(f"Project bundle:   {ROOT / 'data/avo/s01/bundles' / layout.version / 'manifest.json'}")

## Output interpretation

Notebook 01 retains interpreted depth/time horizons as provenance and uses separately named seismic-conformed T6/T7 surfaces as property and elastic modeling boundaries. Reservoir DELTA, P(sand), porosity, Vp, Vs, and density are constructed in the seismic-conformed T6–T7 coordinate.

Three wells (`T732`, `T761`, and `T762`) have complete usable T6/T7 ties and condition the reservoir property model. The exported support and extrapolation masks distinguish well-supported interpolation from lateral extrapolation; pale, low-confidence regions in Section 6 are extrapolated. The source T6/T7 pinch at the end of the line is retained and excluded wherever it violates the configured minimum interval thickness.

The full finite `delta.npy`, `sandprob.npy`, and `poro.npy` arrays are continuous composites: regional outside T6–T7, smoothly tapered at the horizon boundaries, and increasingly reservoir-conditioned toward the supported interval interior. The corresponding `*_reservoir.npy`, `reservoir_mask.npy`, and `property_support_mask.npy` arrays preserve the reservoir-only values, geometry, and support domain.

The Stage-01 contract exposes downstream inputs under `s01data/usable/v001` and `s01data/attributes/v001`. Synthetic realizations and realization-split machine-learning datasets occupy the versioned `s01data/synthetic` and `s01data/datasets` stages.

Elastic outputs now follow the same separation: `elastic_background_regional.npy` is the five-well global-RGT context, `elastic_background_reservoir.npy` is the three-tied-well local-stratigraphic prediction, and `elastic_background.npy` is their smooth confidence-weighted composite.

Use `t6/t7_time_interpreted_ms.npy` for depth-interpretation provenance and `t6/t7_model_seismic_conformed_ms.npy` for the boundaries of generated property and elastic models.
